# LanceDB vector database

In [1]:
import lancedb

# in python script: Path(__file__).parent / "vector_database"
db = lancedb.connect(uri="vector_database")
db

LanceDBConnection(uri='/Users/aigineer/Documents/github/ai_engineering_kokchun_giang/code-alongs/13_lancedb/vector_database')

In [2]:
db.uri

'/Users/aigineer/Documents/github/ai_engineering_kokchun_giang/code-alongs/13_lancedb/vector_database'

## Read in data

In [3]:
import json 
with open("data/animals_text_embeddings.json", "r") as file:
    data = json.loads(file.read())

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

## Create table

In [4]:
db.create_table("animals", exist_ok=True, data=data)


LanceTable(name='animals', version=3, _conn=LanceDBConnection(uri='/Users/aigineer/Documents/github/ai_engineering_kokchun_giang/code-alongs/13_lancedb/vector_database'))

In [5]:
db.list_tables()

ListTablesResponse(tables=['animals'], page_token=None)

In [6]:
db["animals"]

LanceTable(name='animals', version=3, _conn=LanceDBConnection(uri='/Users/aigineer/Documents/github/ai_engineering_kokchun_giang/code-alongs/13_lancedb/vector_database'))

In [7]:
db["animals"].head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

convert to a classic dataframe

In [8]:
df_animals = db["animals"].to_pandas()
df_animals

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [9]:
df_animals.iloc[2]["text"], df_animals.iloc[2]["vector"]

('A large gray elephant drinking water.',
 array([0.88, 0.22, 0.55], dtype=float32))

to add more data

In [10]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

db["animals"].add(more_data)

AddResult(version=4)

In [11]:
db["animals"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Create empty table
- create an empty table first and then place in data 
- need to provide a schema

In [12]:
from lancedb.pydantic import LanceModel

class EmployeeSchema(LanceModel):
    first_name: str 
    last_name: str 
    salary: int

db.create_table(name = "employees", schema=EmployeeSchema, exist_ok=True)

LanceTable(name='employees', version=1, _conn=LanceDBConnection(uri='/Users/aigineer/Documents/github/ai_engineering_kokchun_giang/code-alongs/13_lancedb/vector_database'))

In [13]:
data = [{"first_name": "Bibbi", "last_name": "Babblarna", "salary": 1000}]
db["employees"].add(data)

AddResult(version=2)

In [14]:
db["employees"].to_pandas()

,first_name,last_name,salary
0,Bibbi,Babblarna,1000


In [15]:
db.list_tables()

ListTablesResponse(tables=['animals', 'employees'], page_token=None)

In [16]:
db.drop_table("employees")

In [17]:
db.list_tables()

ListTablesResponse(tables=['animals'], page_token=None)

## Vector search

ANN - approximate nearest neighbour for vector search

1. send in a query vector directly and search
    - this requires that we embed our query first using same embedding as what was used in the knowledge base
2. send in a text and let lancedb automatically embed it and search


In [18]:
db["animals"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [19]:
# assume that we embed our question using same embedding model as the one for animals
# question about elephant
query_vector = [0.9, 0.2, 0.5]

db["animals"].search(query_vector).limit(4).to_pandas()

,text,vector,_distance
0,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]",0.0033
1,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]",0.0094
2,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]",0.0094
3,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]",0.0094


## Embeddings API

- let lancedb embed our documents automatically
- let lancedb embed our query automatically and search using natural language



In [22]:
from lancedb.pydantic import Vector
from lancedb.embeddings import get_registry

model = get_registry().get("gemini-text").create(name="gemini-embedding-001")

model


GeminiText(max_retries=7, name='gemini-embedding-001', query_task_type='retrieval_query', source_task_type='retrieval_document')

In [23]:
embeddings = model.generate_embeddings("Why are SQL good at relationships? Because they are relational")

In [26]:
import numpy as np 
np.array(embeddings).shape

(62, 3072)

In [28]:
class JokeModel(LanceModel):
    joke: str = model.SourceField() # input to embedding function
    embedding: Vector(3072) = model.VectorField() # computed embedding in this column

db.create_table("jokes", schema=JokeModel, exist_ok=True) 

LanceTable(name='jokes', version=1, _conn=LanceDBConnection(uri='/Users/aigineer/Documents/github/ai_engineering_kokchun_giang/code-alongs/13_lancedb/vector_database'))

In [32]:
import pandas as pd 
with open("data/jokes.json", "r") as file:
    jokes_data = json.loads(file.read())

df_jokes = pd.DataFrame(jokes_data).rename({"jokes": "joke"}, axis=1)
df_jokes.head()

,joke
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


In [34]:
db["jokes"].add(df_jokes)

AddResult(version=2)

In [38]:
db["jokes"].to_pandas().head()

,joke,embedding
0,Parallel lines have so much in common—it’s sad...,"[-0.024001757, 0.01247358, -0.024144737, -0.06..."
1,"ETL stands for “Extract, Transform, Leave for ...","[-0.015356114, 0.0211365, -0.021389864, -0.079..."
2,What do you call a snake that runs your script...,"[-0.01761013, 0.0031474787, -0.015632002, -0.0..."
3,"Gold walks into a bar. The bartender says, “Au...","[-0.024867292, 0.013314825, -0.016261652, -0.0..."
4,C# devs don’t argue; they just throw exceptions.,"[-0.0068662865, -0.005415149, 0.0044965413, -0..."


In [42]:
db["jokes"].to_pandas().iloc[2]["embedding"].shape

(3072,)

In [44]:
db["jokes"].search("snakey joke").limit(3).to_pandas()

,joke,embedding,_distance
0,What do you call a snake that runs your script...,"[-0.01761013, 0.0031474787, -0.015632002, -0.0...",0.606556
1,Why did the Python programmer get bitten? Beca...,"[-0.021944793, 0.0030636177, -0.019837778, -0....",0.631211
2,"Gold walks into a bar. The bartender says, “Au...","[-0.024867292, 0.013314825, -0.016261652, -0.0...",0.747921


In [48]:
db["jokes"].search("H2O jokes").limit(3).to_pandas()

,joke,embedding,_distance
0,I told a chemistry joke… there was no reaction.,"[-0.022922393, 0.017959604, -0.029222224, -0.0...",0.708293
1,"Gold walks into a bar. The bartender says, “Au...","[-0.024867292, 0.013314825, -0.016261652, -0.0...",0.709150
2,Why did the chemist ground his kids? Because t...,"[-0.023257235, 0.016145445, -0.029016329, -0.0...",0.729078


## Other search

Term based searching
- tf-idf: Term frequency inverse document frequency 
- BM25

Hybrid search
- combination of term-based and vector


## Reranker

- read yourself